# Food Prediction Model - Version 1

This notebook trains a simple but strong image classifier for Indian food using the `food_dataset` folder.

We keep it beginner-friendly:
- transfer learning with a powerful pretrained model
- light augmentation
- one base-training phase
- one short fine-tuning phase
- save the final model and label list for later use


In [54]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)


TensorFlow version: 2.21.0


In [55]:
# Paths and training settings
DATA_DIR = Path('food_dataset')
IMG_SIZE = (224, 224)  # Good balance for fine textures/details and speed
BATCH_SIZE = 24
SEED = 42
VALIDATION_SPLIT = 0.2
BASE_EPOCHS = 15
FINE_TUNE_EPOCHS = 10

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Could not find the dataset folder: {DATA_DIR.resolve()}")

class_dirs = [p for p in DATA_DIR.iterdir() if p.is_dir()]
class_names = sorted([p.name for p in class_dirs])
num_classes = len(class_names)

print('Dataset found:', DATA_DIR.exists())
print('Number of food classes:', num_classes)
print('Example class names:', class_names[:10])


Dataset found: True
Number of food classes: 106
Example class names: ['aloo_gobi', 'aloo_paratha', 'aloo_tikki', 'andhra_thali', 'appam', 'barfi', 'bengali_thali', 'bhature', 'bhel_puri', 'biryani']


## 1) Load and balance the dataset

We will split images per class, then balance the training set so every food class has about 175 training images after augmentation.

This helps classes with only ~75 images learn much better instead of being drowned out by larger classes.


In [56]:
# Build class-wise file lists first so we can rebalance the training set later.
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
TARGET_TRAIN_IMAGES_PER_CLASS = 175

class_to_paths = {}
train_paths_by_class = {}
val_paths_by_class = {}
train_labels_by_class = {}
val_labels_by_class = {}
class_counts = {}

rng = np.random.default_rng(SEED)

for class_index, class_name in enumerate(class_names):
    paths = [
        p for p in (DATA_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in ALLOWED_EXTENSIONS
    ]
    paths = sorted(paths)
    rng.shuffle(paths)
    class_to_paths[class_name] = paths
    class_counts[class_name] = len(paths)

    split_index = max(1, int(len(paths) * (1.0 - VALIDATION_SPLIT)))
    train_paths = paths[:split_index]
    val_paths = paths[split_index:]
    if len(val_paths) == 0:
        val_paths = paths[-1:]
        train_paths = paths[:-1] if len(paths) > 1 else paths
    
    train_paths_by_class[class_name] = [str(p) for p in train_paths]
    val_paths_by_class[class_name] = [str(p) for p in val_paths]
    train_labels_by_class[class_name] = [class_index] * len(train_paths)
    val_labels_by_class[class_name] = [class_index] * len(val_paths)

num_classes = len(class_names)
min_images = min(class_counts.values())
max_images = max(class_counts.values())
print('Classes:', num_classes)
print('Min images per class:', min_images)
print('Max images per class:', max_images)
print('Target training images per class:', TARGET_TRAIN_IMAGES_PER_CLASS)
print('Example class counts:', list(class_counts.items())[:5])


Classes: 106
Min images per class: 49
Max images per class: 200
Target training images per class: 175
Example class counts: [('aloo_gobi', 200), ('aloo_paratha', 200), ('aloo_tikki', 164), ('andhra_thali', 91), ('appam', 110)]


In [57]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image_and_label(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    label = tf.one_hot(label, num_classes)
    return image, label

def build_class_dataset(paths, labels, training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.repeat()
        ds = ds.take(TARGET_TRAIN_IMAGES_PER_CLASS)
    ds = ds.map(load_image_and_label, num_parallel_calls=AUTOTUNE)
    return ds

def sample_beta_distribution(size, concentration):
    gamma_1 = tf.random.gamma(shape=[size], alpha=concentration)
    gamma_2 = tf.random.gamma(shape=[size], alpha=concentration)
    return gamma_1 / (gamma_1 + gamma_2)

def mixup_batch(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_images = tf.gather(images, indices)
    shuffled_labels = tf.gather(labels, indices)
    lam = sample_beta_distribution(batch_size, alpha)
    lam_x = tf.reshape(lam, (batch_size, 1, 1, 1))
    lam_y = tf.reshape(lam, (batch_size, 1))
    mixed_images = images * lam_x + shuffled_images * (1.0 - lam_x)
    mixed_labels = labels * lam_y + shuffled_labels * (1.0 - lam_y)
    return mixed_images, mixed_labels

def cutmix_single(image, shuffled_image, label, shuffled_label, alpha=1.0):
    lam = sample_beta_distribution(1, alpha)[0]
    cut_ratio = tf.sqrt(1.0 - lam)
    cut_w = tf.cast(tf.cast(IMG_SIZE[1], tf.float32) * cut_ratio, tf.int32)
    cut_h = tf.cast(tf.cast(IMG_SIZE[0], tf.float32) * cut_ratio, tf.int32)
    cx = tf.random.uniform([], 0, IMG_SIZE[1], dtype=tf.int32)
    cy = tf.random.uniform([], 0, IMG_SIZE[0], dtype=tf.int32)
    x1 = tf.clip_by_value(cx - cut_w // 2, 0, IMG_SIZE[1])
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, IMG_SIZE[0])
    x2 = tf.clip_by_value(cx + cut_w // 2, 0, IMG_SIZE[1])
    y2 = tf.clip_by_value(cy + cut_h // 2, 0, IMG_SIZE[0])
    cut_w = tf.maximum(x2 - x1, 1)
    cut_h = tf.maximum(y2 - y1, 1)
    patch = tf.image.crop_to_bounding_box(shuffled_image, y1, x1, cut_h, cut_w)
    patch = tf.image.pad_to_bounding_box(patch, y1, x1, IMG_SIZE[0], IMG_SIZE[1])
    mask = tf.image.pad_to_bounding_box(
        tf.ones((cut_h, cut_w, 3), dtype=image.dtype),
        y1,
        x1,
        IMG_SIZE[0],
        IMG_SIZE[1],
    )
    mixed_image = image * (1.0 - mask) + patch
    area_ratio = tf.cast(cut_w * cut_h, tf.float32) / tf.cast(IMG_SIZE[0] * IMG_SIZE[1], tf.float32)
    mixed_label = label * (1.0 - area_ratio) + shuffled_label * area_ratio
    return mixed_image, mixed_label

def cutmix_batch(images, labels, alpha=1.0):
    batch_size = tf.shape(images)[0]
    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_images = tf.gather(images, indices)
    shuffled_labels = tf.gather(labels, indices)
    mixed_images, mixed_labels = tf.map_fn(
        lambda elems: cutmix_single(*elems, alpha=alpha),
        (images, shuffled_images, labels, shuffled_labels),
        fn_output_signature=(tf.float32, tf.float32),
    )
    return mixed_images, mixed_labels

def apply_batch_augmentation(images, labels):
    choice = tf.random.uniform([])
    return tf.cond(
        choice < 0.5,
        lambda: mixup_batch(images, labels, alpha=0.2),
        lambda: cutmix_batch(images, labels, alpha=1.0),
    )

train_class_datasets = []
val_class_datasets = []

for class_index, class_name in enumerate(class_names):
    train_ds_class = build_class_dataset(
        train_paths_by_class[class_name],
        train_labels_by_class[class_name],
        training=True,
    )
    train_class_datasets.append(train_ds_class)

    val_ds_class = tf.data.Dataset.from_tensor_slices(
        (val_paths_by_class[class_name], val_labels_by_class[class_name])
    )
    val_ds_class = val_ds_class.map(load_image_and_label, num_parallel_calls=AUTOTUNE)
    val_class_datasets.append(val_ds_class)

train_ds = train_class_datasets[0]
for ds in train_class_datasets[1:]:
    train_ds = train_ds.concatenate(ds)

train_ds = train_ds.shuffle(5000, seed=SEED, reshuffle_each_iteration=True)
train_ds_clean = train_ds.batch(BATCH_SIZE, drop_remainder=True)
train_ds_clean = train_ds_clean.prefetch(buffer_size=AUTOTUNE)

# Stronger version with MixUp/CutMix, used only for fine-tuning.
train_ds_strong = train_ds.batch(BATCH_SIZE, drop_remainder=True)
train_ds_strong = train_ds_strong.map(apply_batch_augmentation, num_parallel_calls=AUTOTUNE)
train_ds_strong = train_ds_strong.prefetch(buffer_size=AUTOTUNE)

val_ds = val_class_datasets[0]
for ds in val_class_datasets[1:]:
    val_ds = val_ds.concatenate(ds)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

print('Balanced training datasets are ready.')
print('Base training uses clean batches; fine-tuning uses MixUp/CutMix.')
print('Training will sample about', TARGET_TRAIN_IMAGES_PER_CLASS, 'images per class before mixup/cutmix.')


Balanced training datasets are ready.
Base training uses clean batches; fine-tuning uses MixUp/CutMix.
Training will sample about 175 images per class before mixup/cutmix.


## 2) Build the model

We use ConvNeXtTiny because it is strong on fine-grained visual cues like texture, shape, and color.

The balanced base training set learns the clean signal first, then MixUp/CutMix is used later to strengthen fine-tuning.

In [58]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.12),
        layers.RandomContrast(0.1),
        layers.RandomTranslation(0.04, 0.04),
        layers.RandomBrightness(0.04),
    ],
    name='data_augmentation',
)

base_model = keras.applications.ConvNeXtTiny(
    include_top=False,
    weights='imagenet',
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
# ConvNeXt expects standardized inputs around [-1, 1].
x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(512, activation='gelu')(x)
x = layers.Dropout(0.25)(x)
x = layers.Dense(256, activation='gelu')(x)
x = layers.Dropout(0.15)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()


Model: "functional_47"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_54 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_7 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 7, 7, 768)      │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 106)            │        27,242 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,372,426 (108.23 MB)

 Trainable params: 552,298 (2.11 MB)

 Non-trainable params: 27,820,128 (106.13 MB)

In [59]:
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
)


In [60]:
# Optional quick sanity check (tiny run) before full training
# Extract a single pre-batched batch as numpy arrays and run a quick fit to avoid dataset iterator issues
batch_iter = iter(train_ds_clean.take(1))
images, labels = next(batch_iter)
# convert to numpy (eager tensors)
try:
    x_batch = images.numpy()
    y_batch = labels.numpy()
except Exception:
    x_batch = tf.keras.backend.get_value(images)
    y_batch = tf.keras.backend.get_value(labels)

val_iter = iter(val_ds.take(1))
val_images, val_labels = next(val_iter)
try:
    x_val = val_images.numpy()
    y_val = val_labels.numpy()
except Exception:
    x_val = tf.keras.backend.get_value(val_images)
    y_val = tf.keras.backend.get_value(val_labels)

print('x_batch.shape, y_batch.shape, x_val.shape, y_val.shape:', x_batch.shape, y_batch.shape, x_val.shape, y_val.shape)
print('dtypes:', x_batch.dtype, y_batch.dtype, x_val.dtype, y_val.dtype)

# Recompile to run eagerly for the smoke test to avoid tf.function iterator issues in this environment
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
    run_eagerly=True,
)

smoke_history = model.fit(
    x_batch,
    y_batch,
    validation_data=(x_val, y_val),
    epochs=1,
    verbose=1,
 )

# Recompile back to graph mode for full training
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
)


x_batch.shape, y_batch.shape, x_val.shape, y_val.shape: (24, 224, 224, 3) (24, 106) (24, 224, 224, 3) (24, 106)
dtypes: float32 float32 float32 float32
1/1 ━━━━━━━━━━━━━━━━━━━━ 45s 45s/step - accuracy: 0.0000e+00 - loss: 5.0645 - top5_acc: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 4.9505 - val_top5_acc: 0.0000e+00


## 3) Train the model

The model first learns from clean balanced batches so early accuracy is stable, then we switch to MixUp/CutMix for harder fine-tuning.

That keeps the first epoch from bouncing around while still forcing the network to learn texture and color details later.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        mode='max',
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.TerminateOnNaN(),
]

history = model.fit(
    train_ds_clean,
    validation_data=val_ds,
    epochs=max(5, BASE_EPOCHS - 5),
    callbacks=callbacks,
    verbose=2,
)

history_strong = model.fit(
    train_ds_strong,
    validation_data=val_ds,
    epochs=BASE_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)


Epoch 1/10


## 4) Fine-tune the last layers

This second pass can improve accuracy a lot while still keeping the notebook simple.

In [ ]:
base_model.trainable = True

# Unfreeze only the last part of the backbone and keep normalization layers frozen.
for layer in base_model.layers[:-80]:
    layer.trainable = False

for layer in base_model.layers[-80:]:
    if isinstance(layer, (layers.BatchNormalization, layers.LayerNormalization)):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-4),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
)

fine_tune_history = model.fit(
    train_ds_strong,
    validation_data=val_ds,
    epochs=BASE_EPOCHS + FINE_TUNE_EPOCHS,
    initial_epoch=len(history.history['loss']) + len(history_strong.history['loss']),
    callbacks=callbacks,
    verbose=2,
)


## 5) Save what we need for later

We save the trained model, the class names, and the history so the model can be reused without training again.


In [ ]:
with open('class_names.pkl', 'wb') as f:
    pickle.dump(class_names, f)

with open('history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

with open('fine_tune_history.pkl', 'wb') as f:
    pickle.dump(fine_tune_history.history, f)

print('Saved best_model.keras, class_names.pkl, history.pkl, and fine_tune_history.pkl')


## 6) Load the saved model and predict one image

This is the small reusable part you can copy into your app later.


In [ ]:
loaded_model = keras.models.load_model('best_model.keras')

with open('class_names.pkl', 'rb') as f:
    loaded_class_names = pickle.load(f)


def predict_food(image_path):
    img = keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = loaded_model.predict(img_array, verbose=0)
    predicted_index = np.argmax(predictions[0])
    predicted_label = loaded_class_names[predicted_index]
    confidence = float(np.max(predictions[0]))

    return predicted_label, confidence

# Example:
# label, confidence = predict_food('some_image.jpg')
# print(label, confidence)


In [ ]:
import random

# Random image demo: show the real label and the model prediction
all_image_paths = [
    p for p in DATA_DIR.glob('*/*')
    if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
 ]

random_image_path = random.choice(all_image_paths)
actual_label = random_image_path.parent.name

predicted_label, confidence = predict_food(str(random_image_path))

img = keras.utils.load_img(random_image_path, target_size=IMG_SIZE)
plt.figure(figsize=(7, 7))
plt.imshow(img)
plt.axis('off')
plt.title(f'Actual: {actual_label} | Predicted: {predicted_label} ({confidence:.2%})')
plt.show()

print('Image path   :', random_image_path)
print('Actual label :', actual_label)
print('Predicted    :', predicted_label)
print(f'Confidence   : {confidence:.4f}')
